<a href="https://colab.research.google.com/github/ailingomezromay/Grupo-4--TP-BigData-Burnout/blob/main/TP_Grupo%204_BigData_Burnout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Introducción y elección del dataset

# TP Final - Big Data y MLOps
Predicción de riesgo de burnout en empleados del sector tecnológico

# Alumnos / Grupo 4
Ailin Gomez Romay  
Carlos Lazzarino  
Victoria Reppucci  
Vanessa Vidal

# Objetivo del trabajo

El objetivo de este trabajo es desarrollar un modelo de Machine Learning que permita predecir el riesgo de burnout en empleados del sector tecnológico, a partir de variables demográficas, laborales y relacionadas con la salud mental.

El dataset fue obtenido de Kaggle ( https://www.kaggle.com/datasets/suhanigupta04/employee-mental-health-and-burnout-dataset ).

Contiene registros sintéticos de empleados del sector tech. Lo elegimos porque el volumen justifica el uso de PySpark y las variables tienen sentido intuitivo: horas trabajadas, nivel de estrés, calidad de sueño.

El problema a resolver consiste en identificar qué empleados presentan mayor riesgo de burnout en función de distintas variables observables. Este tipo de análisis puede resultar útil para áreas de recursos humanos y bienestar organizacional, ya que permite anticipar situaciones de riesgo y tomar decisiones preventivas.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import mlflow
import mlflow.spark
import shap
from sklearn.linear_model import LogisticRegression as SklearnLR

ModuleNotFoundError: No module named 'mlflow'

# 1. Carga del dataset

En esta sección se carga el dataset seleccionado utilizando la funcionalidad de subida de archivos de Google Colab.

Una vez cargado, se realiza una inspección inicial para verificar la cantidad de registros y variables disponibles. El dataset cuenta con aproximadamente 150.000 observaciones y 25 variables, lo que permite trabajar en un contexto cercano a Big Data.

In [ ]:
data_path = 'tech_mental_health_burnout.csv'

df = pd.read_csv(data_path)

print(df.shape)
df.head()

In [ ]:
# 2. Información general
df.info()

# Exploración inicial del dataset

El dataset contiene información sobre empleados del sector tecnológico, incluyendo variables demográficas, laborales y de salud mental. A partir de una primera inspección, se observa que la mayoría de las variables son numéricas, aunque también hay variables categóricas que deberán ser transformadas para el modelado.

# 3. Análisis de la variable objetivo

## Desbalance de clases

Se observa un fuerte desbalance en la variable `burnout_level`, con predominio de la categoría "Low" y muy pocos casos en "High".

Este desbalance puede afectar el desempeño del modelo, ya que los algoritmos tienden a favorecer la clase mayoritaria.

Por este motivo, se decide reformular el problema como una clasificación binaria, agrupando "Moderate" y "High" en una única categoría de riesgo.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="burnout_level")
plt.title("Distribución de burnout_level")
plt.xlabel("Nivel de burnout")
plt.ylabel("Cantidad de empleados")
plt.show()

## Distribución de la variable objetivo

El gráfico confirma el desbalance. La clase 'Low' representaba aproximadamente el 88% de los registros. Por eso decidimos binarizar, uniendo 'Moderate' y 'High' en una sola clase de riesgo. No aplicamos técnicas de balanceo como SMOTE o class_weight por limitaciones de tiempo, aunque queda pendiente para una versión futura.

# 4. Relación con variables clave

## Estrés

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="burnout_level", y="stress_level")
plt.title("Estrés según nivel de burnout")
plt.show()

## Relación entre estrés y burnout

La separación entre grupos es bastante pronunciada en el caso del estrés, más que en otras variables.

## Horas trabajadas

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="burnout_level", y="work_hours_per_week")
plt.title("Horas trabajadas según burnout")
plt.show()

## Relación entre horas trabajadas y burnout

La diferencia no es dramática pero la tendencia es consistente: más horas, más burnout.

## Sueño

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="burnout_level", y="sleep_hours")
plt.title("Horas de sueño según burnout")
plt.show()

## Relación entre sueño y burnout

Lo interesante es que la diferencia en horas de sueño entre 'Low' y 'High' es menor de lo que esperábamos. Igualmente la tendencia se mantiene.

# 5. Correlaciones

In [ ]:
cols = [
    "work_hours_per_week", "overtime_hours", "job_satisfaction",
    "manager_support", "work_life_balance", "sleep_hours",
    "stress_level", "anxiety_score", "depression_score", "burnout_score"
]

corr = df[cols].corr()

plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Matriz de correlación")
plt.show()

print(df.isnull().sum().sum())  # total nulos
for col in ["gender", "job_role", "company_size", "work_mode"]:
    print(col, df[col].nunique())

## Análisis de correlaciones

A partir de la matriz de correlación se observa que variables relacionadas con la salud mental, como el nivel de estrés, ansiedad y depresión, presentan una alta correlación positiva con el burnout_score.

En particular, el estrés muestra la mayor correlación, lo que sugiere que es uno de los factores más determinantes del burnout.

Por otro lado, variables como el balance entre vida laboral y personal y las horas de sueño presentan correlaciones negativas, lo que indica que podrían actuar como factores protectores.

El dataset no presentaba valores nulos. Las variables categóricas tenían las siguientes cardinalidades: gender: 3, job_role: 8, company_size: 4, work_mode: 3.

## 6. Modelado con PySpark

En esta sección se entrenan distintos modelos de Machine Learning utilizando PySpark.

Se construye un pipeline que incluye la transformación de variables y el entrenamiento del modelo, permitiendo comparar distintos algoritmos y evaluar su desempeño.

In [ ]:
spark = SparkSession.builder \
    .appName("TP_BigData_Burnout") \
    .getOrCreate()

print(spark)

df_spark = spark.createDataFrame(df)
df_spark.printSchema()

In [ ]:
df_spark = df_spark.withColumn(
    "burnout_risk",
    F.when(F.col("burnout_level").isin("Moderate", "High"), 1).otherwise(0)
)

df_spark = df_spark.drop("burnout_level", "burnout_score")

df_spark.printSchema()

## Preparación de datos

Se realizó la transformación de variables categóricas mediante StringIndexer y OneHotEncoder, permitiendo su uso en modelos de Machine Learning.

Las variables numéricas no tenían nulos ni outliers extremos, por eso las dejamos sin transformación adicional. Sí evaluamos si tenía sentido normalizar, pero dado que usamos modelos basados en árboles y regresión logística con regularización, no era estrictamente necesario.

Se construyó un pipeline utilizando PySpark para automatizar el proceso de transformación y facilitar la reproducibilidad del modelo.

#6.1. Pipeline completa

In [ ]:
categorical_cols = [
    "gender", "job_role", "company_size", "work_mode"
]

numeric_cols = [
    "age", "experience_years", "work_hours_per_week", "overtime_hours",
    "meetings_per_day", "deadlines_missed", "job_satisfaction",
    "manager_support", "work_life_balance", "sleep_hours",
    "physical_activity_days", "screen_time_hours", "caffeine_intake",
    "social_support_score", "has_therapy", "stress_level",
    "anxiety_score", "depression_score", "seeks_professional_help"
]

target_col = "burnout_risk"

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe")
    for col in categorical_cols
]

feature_cols = numeric_cols + [f"{col}_ohe" for col in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_df, test_df = df_spark.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_df.count())
print("Test:", test_df.count())

Una limitación del pipeline actual es que el StringIndexer se fittea sobre todo el dataframe antes del split, lo que podría introducir data leakage si hay categorías poco frecuentes. En una versión productiva esto se resolvería fiteando el indexer solo sobre train.

#6.2 Logistic Regression

In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="burnout_risk"
)

pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, lr])

model_lr = pipeline_lr.fit(train_df)
predictions_lr = model_lr.transform(test_df)

## Modelo 1: Logistic Regression

Como primer modelo se utiliza regresión logística, ya que constituye una línea de base sólida para problemas de clasificación binaria.

Además de ser un modelo interpretable, permite obtener una primera referencia de desempeño antes de probar algoritmos más complejos.

#6.3 Evaluacion

In [ ]:
evaluator = BinaryClassificationEvaluator(
    labelCol="burnout_risk",
    metricName="areaUnderROC"
)

auc_lr = evaluator.evaluate(predictions_lr)

print("AUC Logistic Regression:", auc_lr)

## Evaluación del modelo

Para evaluar el desempeño del modelo se utiliza la métrica AUC (Area Under the ROC Curve), ya que resulta adecuada para problemas de clasificación binaria y permite medir la capacidad del modelo para distinguir entre clases.

## Limitaciones del modelo

Una limitación importante es el desbalance de clases: con el 88% en la clase mayoritaria, el AUC puede sobreestimar el desempeño real. Por eso complementamos con F1 y la matriz de confusión más adelante.

#6.4 RandomForest

In [ ]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="burnout_risk",
    numTrees=50,
    seed=42
)

pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])

model_rf = pipeline_rf.fit(train_df)
predictions_rf = model_rf.transform(test_df)

#6.5 Evaluacion

In [ ]:
auc_rf = evaluator.evaluate(predictions_rf)
print("AUC Random Forest:", auc_rf)

## Comparación de modelos

Elegimos Logistic Regression como modelo final por dos razones: tuvo mejor AUC (0.962 vs 0.927) y permite analizar los coeficientes para entender qué variables pesan más. Igual, un AUC de 0.962 nos generó dudas es un número muy alto. Revisamos que burnout_score no estuviera filtrando el target y confirmamos que la habíamos eliminado antes del entrenamiento. Aun así, con clases desbalanceadas el AUC puede ser engañoso.

In [ ]:
print("Logistic Regression AUC:", auc_lr)
print("Random Forest AUC:", auc_rf)


Se calcula `f1_lr` y `f1_rf` aquí, antes de usarlas en MLflow. Este era el error principal que causaba `NameError: name 'f1_lr' is not defined`.

In [ ]:

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="burnout_risk",
    metricName="f1"
)

f1_lr = evaluator_f1.evaluate(predictions_lr)
f1_rf = evaluator_f1.evaluate(predictions_rf)

print("F1 Logistic Regression:", f1_lr)
print("F1 Random Forest:", f1_rf)

In [ ]:
# Extraer predicciones para matriz de confusión
preds = predictions_lr.select("burnout_risk", "prediction").toPandas()
cm = confusion_matrix(preds["burnout_risk"], preds["prediction"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Sin riesgo", "En riesgo"])
disp.plot()
plt.title("Matriz de confusión - Logistic Regression")
plt.show()

Para complementar el AUC calculamos el F1-score (0.93) y la matriz de confusión. Esto es especialmente importante dado el desbalance: el F1 penaliza los falsos negativos, que en este contexto son los casos de burnout que el modelo no detecta.

# MLflow

In [ ]:
mlflow.set_experiment("TP_BigData_Burnout")

Registrar Logistic

In [ ]:

mlflow.end_run()

with mlflow.start_run(run_name="Logistic_Regression") as run_lr:
    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_metric("AUC", auc_lr)
    mlflow.log_metric("F1", f1_lr)


    mlflow.spark.log_model(model_lr, "model")
    run_id_lr = run_lr.info.run_id


model_uri = f"runs:/{run_id_lr}/model"
mlflow.register_model(model_uri, "Burnout_Model")

Registrar Random Forest

In [ ]:
with mlflow.start_run(run_name="Random_Forest"):
    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_metric("AUC", auc_rf)
    mlflow.log_metric("F1", f1_rf)

## Registro de experimentos con MLflow

Usamos MLflow para registrar los dos modelos y poder comparar métricas desde la UI. Además de AUC logueamos también F1 y los parámetros principales de cada modelo.

# SHAP - Interpretabilidad del modelo final

In [ ]:
# Convertir a pandas para usar SHAP
train_pd = train_df.select(numeric_cols + ["burnout_risk"]).toPandas().dropna()
X_train = train_pd[numeric_cols]
y_train = train_pd["burnout_risk"]

sk_model = SklearnLR(max_iter=1000)
sk_model.fit(X_train, y_train)

explainer = shap.LinearExplainer(sk_model, X_train)
shap_values = explainer.shap_values(X_train)

shap.summary_plot(shap_values, X_train, plot_type="bar",
                  title="Importancia de variables - SHAP")

Usamos SHAP para analizar qué variables tienen más peso en el modelo. Esto valida la elección de Logistic Regression: no solo tiene mejor AUC sino que podemos explicar sus predicciones, lo cual es importante si el área de RRHH va a usar esto para tomar decisiones.

Según el gráfico SHAP, las variables con mayor impacto fueron stress_level, work_hours_per_week y work_life_balance, lo cual es consistente con lo observado en el EDA.

# Ejemplo de inferencia

In [ ]:
predictions_lr.select("burnout_risk", "prediction", "probability").show(10, truncate=False)

ejemplo = spark.createDataFrame([{
    "age": 32, "gender": "Male", "job_role": "Engineer",
    "company_size": "Large", "work_mode": "Remote",
    "experience_years": 5, "work_hours_per_week": 50,
    "overtime_hours": 10, "meetings_per_day": 4,
    "deadlines_missed": 2, "job_satisfaction": 3,
    "manager_support": 2, "work_life_balance": 2,
    "sleep_hours": 5, "physical_activity_days": 1,
    "screen_time_hours": 9, "caffeine_intake": 4,
    "social_support_score": 3, "has_therapy": 0,
    "stress_level": 8, "anxiety_score": 7,
    "depression_score": 6, "seeks_professional_help": 0
}])

prediccion = model_lr.transform(ejemplo)
prediccion.select("prediction", "probability").show(truncate=False)

El modelo predijo burnout_risk = 1 (en riesgo) con una probabilidad del 99%. Para un perfil con 50 horas semanales, estrés 8/10 y poco descanso, el resultado tiene sentido y muestra que el modelo discrimina bien en casos extremos. Así es como podría usarse en la práctica: RRHH ingresa el perfil de un empleado y obtiene una señal de alerta antes de que el problema escale.

## Conclusiones finales

En este trabajo se desarrolló un modelo de Machine Learning para predecir el riesgo de burnout en empleados del sector tecnológico.

El análisis exploratorio permitió identificar variables relevantes como el nivel de estrés, las horas trabajadas y las horas de sueño, las cuales mostraron una relación significativa con el burnout.

Se implementó un pipeline de procesamiento utilizando PySpark, y se entrenaron modelos de clasificación evaluados mediante la métrica AUC. Además, se utilizó MLflow para registrar los experimentos y asegurar la reproducibilidad del proceso.

A partir de la comparación de modelos, se seleccionó Logistic Regression como modelo final debido a su mejor desempeño y mayor interpretabilidad.

Lo que más nos costó fue entender el efecto del desbalance de clases en las métricas. Al principio el AUC parecía excelente, pero al revisar las predicciones individuales notamos que el modelo tendía a predecir siempre la clase mayoritaria. Eso nos hizo replantear cómo interpretar los resultados. Para una próxima versión aplicaríamos balanceo de clases y probablemente ajustaríamos el threshold de decisión en lugar de usar 0.5 por defecto.